# Semantic Search: Find sentences that are similar to an asked question

This notebook looks into symmetric and asymmetric search methods. These methods are tested with questions that should give a reasonable answer and questions it should not know an answer to.

Our semantic search is an asymmetric search task. Our questions are likely to be shorter than the sentences prompt back to us. Which was, in case of `corpus_free_3600_250606.csv`, 24 words per sentence on average.

(See `explore_corpus.ipynb` for the average number of words per sentence and other descriptive statistical properties.)

It appears that a symmetric approach can be optimised for a asymmetric query by using a cross-encoder for the top_k number of sentences.

### Settings

In [ ]:
# Settings
embedding_file = "embedding_corpus_free_3600_250606.pickle"
raw_corpus_file = "corpus_free_3600_250606.csv"
used_st_model = "NeuML/pubmedbert-base-embeddings"
cs_model = 'cross-encoder/ms-marco-MiniLM-L6-v2'

# Number of sentences returned. (For proposed optimalisation of SBERT)
top_k = 35


"msmarco-distilbert-dot-v5"

### Initialisation

#### Import and functions

In [ ]:
# Import
import pickle
import pandas as pd
from sentence_transformers import SentenceTransformer
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers import util

In [ ]:
# Functions
def preprocess_query(query, embedding_model):
    """Natural query goes in, preprocessed (lowercasing, remove punctuations) and embedded query goes out."""
    
    # Pre-processing
    query = query.lower().strip('\[.*?\]')
    # NOTE: Lemmatization need to be applied if this was also applied to corpus_embedding.

    # Encode query
    query = embedding_model.encode(query)

    return query

def symmetric_query(query_embedding, corpus_embedding, corpus_df: pd.Series, top_k: int = 10):
    """df_corpus: Series containing sentences and an index that corresponds with the sentence_id (pd.Series)"""
    # semantic_search uses the 'exact nearest neighbor' algorithm
    # https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html#approximate-nearest-neighbor
    # symmetric semantic search
    result = util.semantic_search(
        query_embeddings= query_embedding,
        corpus_embeddings= corpus_embedding,
        top_k= top_k,
    )

    # Place results in a data frame
    df = pd.DataFrame(
        {
            'sentence_id': [it['corpus_id'] for it in result[0]], # sentence_id of cos_sim with highest score.
            'cos_sim': [it['score'] for it in result[0]] # cos_sim score of retrieved sentence-query.
        }
    )

    # Merge with corpus data frame
    df = pd.merge(df, corpus_df, left_on= 'sentence_id', right_index=True)

    return df

#### Load corpus, vectors and used transformer

In [ ]:
# Load raw corpus
df_corpus = pd.read_csv(f'../../data/corpus/{raw_corpus_file}')

In [ ]:
# Load sentence embeddings
with open(f"../../data/vectors/{embedding_file}", 'rb') as handle:
    embeddings = pickle.load(handle)

In [ ]:
# Load used transformer
model_st = SentenceTransformer(used_st_model)
model_cs = CrossEncoder(cs_model)

### Asymmetric search

TODO: have a look at https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html#question-answer-retrieval

I'm a bit confused.. SBERT states that it has a method called .encode_query and .encode_document, though the pre-trained model they are referring to does not have this option.

In [ ]:
model_msmarco = SentenceTransformer("msmarco-distilbert-dot-v5")

In [ ]:
# model_msmarco.encode_document()

### Symmetric search

Note that if the embedding contains more than one million rows a different search method is needed. `util.semantic_search` uses exact nearest neighbor which in that case would be time consuming. It is advised to use approximate nearest neighbor then instead.

https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html#approximate-nearest-neighbor

In [ ]:
# Check if our matrix contains more than 1.000.000 rows:
embeddings.shape

Proposed optimalisation by re-ranking with cross-encoders: https://github.com/UKPLab/sentence-transformers/tree/master/examples/sentence_transformer/applications/retrieve_rerank

#### Query from my thesis

In [ ]:
# Query from my thesis
query = "How do pesticides affects the human body and give rise to Parkinson's disease?"

results = symmetric_query(
    query_embedding= preprocess_query(
        query,
        model_st
    ),
    corpus_embedding= embeddings,
    corpus_df= df_corpus['sentence_text'],
    top_k= top_k
)
results.head(10)

In [ ]:
# Without optimalisation
results['sentence_text'].values[:3]

In [ ]:
# Optimise with cross-encoder
cross_inp = [[query, hit] for hit in results['sentence_text'].values]
cross_scores = model_cs.predict(cross_inp)

results['cross_scores'] = cross_scores

results = results.sort_values(by='cross_scores', ascending=False)


In [ ]:
# With optimalisation
results['sentence_text'].values[:3]

#### Is Parkinson's disease hereditary?

In [ ]:
# Query
query = "Is Parkinson's disease hereditary?"

results = symmetric_query(
    query_embedding= preprocess_query(
        query,
        model_st,
    ),
    corpus_embedding= embeddings,
    corpus_df= df_corpus['sentence_text'],
    top_k= top_k,
)
results.head(10)

In [ ]:
results['sentence_text'].values[:3]

In [ ]:
# Optimise with cross-encoder
cross_inp = [[query, hit] for hit in results['sentence_text'].values]
cross_scores = model_cs.predict(cross_inp)

results['cross_scores'] = cross_scores

results = results.sort_values(by='cross_scores', ascending=False)
results.head(10)

In [ ]:
# With optimalisation
results['sentence_text'].values[:3]

#### "Negative control" question: What is the capital of The Netherlands?

In [ ]:
# Query
query = "How many people live in The Netherlands?"

results = symmetric_query(
    query_embedding= preprocess_query(
        query,
        model_st
    ),
    corpus_embedding= embeddings,
    corpus_df= df_corpus['sentence_text'],
    top_k= top_k
)
results.head(10)

In [ ]:
results['sentence_text'].values[:3]

In [ ]:
# Optimise with cross-encoder
cross_inp = [[query, hit] for hit in results['sentence_text'].values]
cross_scores = model_cs.predict(cross_inp)

results['cross_scores'] = cross_scores

results = results.sort_values(by='cross_scores', ascending=False)
results.head(10)

In [ ]:
results['sentence_text'].values[:3]

#### "Negative control" question: What is the answer to the Great Question, of Life, the Universe and Everything? (Douglas Adams)

In [ ]:
# Query
query = "What is the answer to the Great Question, of Life, the Universe and Everything?"

results = symmetric_query(
    query_embedding= preprocess_query(
        query,
        model_st
    ),
    corpus_embedding= embeddings,
    corpus_df= df_corpus['sentence_text'],
    top_k=top_k
)
results.head(10)

In [ ]:
results['sentence_text'].values[:3]

In [ ]:
# Optimise with cross-encoder
cross_inp = [[query, hit] for hit in results['sentence_text'].values]
cross_scores = model_cs.predict(cross_inp)

results['cross_scores'] = cross_scores

results = results.sort_values(by='cross_scores', ascending=False)
results.head(10)

In [ ]:
results['sentence_text'].values[:3]